# Calculate country level population data

In [ ]:
import os
import xarray as xr
import numpy as np

In [ ]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
MASKS_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

In [ ]:
# Use country mask to allign population data
mask_file = "GBD_Country_Masks_0.10.nc"
mask_path = os.path.join(MASKS_DIR, mask_file)
masks = xr.open_dataarray(mask_path)

pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
pop = xr.open_dataarray(pop_path)

# Adjust indices to match (with small tolerance)
# e.g., max 1e-7 km distance
pop = pop.reindex_like(masks, method="nearest", tolerance=1e-9)

In [ ]:
# Loop over countries, sum the population for each country and apply to list
population_by_country = []
for i in range(len(masks.country)):
    print(masks.isel(country=i)["country"].values)
    mask = masks.isel(country=i)
    country = masks.isel(country=i)["country"]
    pop_country = (xr.where(
        mask == 1,
        pop,
        np.nan)).sum(dim=("lat", "lon"))
    population_by_country.append(pop_country)

pop_array = xr.concat(population_by_country, "country")

In [ ]:
# Save country level population
description = ("Country level population sum for years 2000-2100 "
               "- scripts by A.F. Wells (2025)")

pop_array.attrs["description"] = description

out_file = "ssp2_country_level_2000-2100.nc"
out_path = os.path.join(POP_DIR, out_file)
pop_array.to_netcdf(out_path)